In [ ]:
import os
import pandas as pd
import json
import random
from collections import defaultdict

PROJECT_NAME = "adam_and_eve"
# None to include all experiments for the project, or list of names e.g. ["opening_5"]
EXPERIMENT_NAMES = None

BASE_MODEL = "gpt-4o-mini-2024-07-18"
MODEL_RECORDS = "model_records_db.md"
LABELS_PATH = "labels/generate_writing/"
TRAINING_FILENAME = "temp/dpo_generator_training"
DPO_BETA = 0.1
# How many bad examples to pair with each amazing example (balances dataset vs diversity)
MAX_PAIRS_PER_CHOSEN = 5
# Allow pairing chosen/rejected from different batches (different prompts) as fallback
ALLOW_CROSS_BATCH_PAIRS = True

In [ ]:
# Load all labeled data for the project
all_files = sorted([
    f for f in os.listdir(LABELS_PATH)
    if PROJECT_NAME in f and f.endswith(".parquet")
])

if EXPERIMENT_NAMES is None:
    file_names = all_files
else:
    file_names = [
        f for f in all_files
        if any(f"{PROJECT_NAME}-{exp}" in f for exp in EXPERIMENT_NAMES)
    ]

print(f"Loading {len(file_names)} label files")
dfs = [pd.read_parquet(LABELS_PATH + fn) for fn in file_names]
df = pd.concat(dfs, ignore_index=True)
print(f"Total labeled examples: {len(df)}")
print(df["label"].value_counts().to_string())

In [ ]:
# Group examples by batch_id (same batch = same prompt)
# Use 'text' (original generation) since 'target_text' is empty for bad examples
groups = defaultdict(lambda: {"amazing": [], "ok": [], "bad": []})

for _, row in df.iterrows():
    label = row["label"]
    if label not in ["amazing", "ok", "bad"]:
        continue
    # For amazing/ok, prefer edited target_text; for bad, use original text
    if label in ["amazing", "ok"] and row.get("target_text", ""):
        content = row["target_text"]
    else:
        content = row["text"]
    if content:
        groups[row["batch_id"]][label].append(content)

print(f"Batches with labels: {len(groups)}")
for batch_id, examples in list(groups.items())[:3]:
    print(f"  {batch_id}: amazing={len(examples['amazing'])}, ok={len(examples['ok'])}, bad={len(examples['bad'])}")

In [ ]:
# Load system and user prompts for each batch
batch_prompts = {}
for batch_id in groups:
    base = f"generated_text/{batch_id}"
    user_path = os.path.join(base, "user_prompt.txt")
    sys_path = os.path.join(base, "system_prompt.txt")
    if not os.path.exists(user_path):
        print(f"Warning: missing prompt for batch {batch_id!r}, skipping")
        continue
    with open(user_path) as f:
        user_prompt = f.read()
    sys_prompt = ""
    if os.path.exists(sys_path):
        with open(sys_path) as f:
            sys_prompt = f.read()
    batch_prompts[batch_id] = (sys_prompt, user_prompt)

print(f"Loaded prompts for {len(batch_prompts)} batches")

In [ ]:
# Create DPO pairs within batches (same prompt)
# Chosen = amazing (or ok as fallback), Rejected = bad
dpo_pairs = []

for batch_id, examples in groups.items():
    if batch_id not in batch_prompts:
        continue
    sys_prompt, user_prompt = batch_prompts[batch_id]
    chosen = examples["amazing"] or examples["ok"]
    rejected = examples["bad"]
    if not chosen or not rejected:
        continue
    for c in chosen:
        sample_size = min(MAX_PAIRS_PER_CHOSEN, len(rejected))
        for r in random.sample(rejected, sample_size):
            dpo_pairs.append({"system": sys_prompt, "user": user_prompt, "chosen": c, "rejected": r})

print(f"Intra-batch pairs: {len(dpo_pairs)}")

if ALLOW_CROSS_BATCH_PAIRS and len(dpo_pairs) < 10:
    print("Too few intra-batch pairs — creating cross-batch pairs")
    all_chosen = []
    all_rejected = []
    for batch_id, examples in groups.items():
        if batch_id not in batch_prompts:
            continue
        sys_p, user_p = batch_prompts[batch_id]
        for c in examples["amazing"] + examples["ok"]:
            all_chosen.append({"system": sys_p, "user": user_p, "text": c})
        for r in examples["bad"]:
            all_rejected.append({"system": sys_p, "user": user_p, "text": r})

    random.shuffle(all_rejected)
    for c_item in all_chosen:
        for r_item in random.sample(all_rejected, min(MAX_PAIRS_PER_CHOSEN, len(all_rejected))):
            dpo_pairs.append({
                "system": c_item["system"],
                "user": c_item["user"],
                "chosen": c_item["text"],
                "rejected": r_item["text"],
            })
    print(f"Total pairs after cross-batch: {len(dpo_pairs)}")

print(f"\nFinal DPO pair count: {len(dpo_pairs)}")

In [ ]:
# ---- OPTIONAL: Bootstrap pairs from source text agent (2_source_text_agent.ipynb) ----
# Run this cell to ADD source-text pairs to whatever is already in dpo_pairs.
# Run it ALONE (skipping cells 1-4) if you have no human labels yet.
#
# Chosen  = passage you approved in the source text agent
# Rejected = base model generation for the same story prompt (auto-generated here)

SOURCE_SECTIONS_FILE = f"finetuning_data/{PROJECT_NAME}/source_sections.jsonl"

if not os.path.exists(SOURCE_SECTIONS_FILE):
    print(f"No source sections at {SOURCE_SECTIONS_FILE} — skipping bootstrap")
    print("Run 2_source_text_agent.ipynb first to produce this file.")
else:
    from openai import OpenAI
    from tqdm.notebook import tqdm

    openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    # Load story prompts
    prompt_dir = f"source_prompts/{PROJECT_NAME}/"
    sys_prompt = ""
    sys_path = os.path.join(prompt_dir, "system_prompt.txt")
    if os.path.exists(sys_path):
        with open(sys_path) as f:
            sys_prompt = f.read()
    user_prompts = []
    for fn in sorted(os.listdir(prompt_dir)):
        if fn.endswith(".txt") and fn != "system_prompt.txt":
            with open(os.path.join(prompt_dir, fn)) as f:
                user_prompts.append(f.read())
    if not user_prompts:
        raise ValueError(f"No user prompts found in {prompt_dir}")

    # Load chosen passages
    with open(SOURCE_SECTIONS_FILE) as f:
        source_sections = [json.loads(line) for line in f if line.strip()]
    chosen_sections = [s for s in source_sections if s.get("type") == "chosen"]
    print(f"Found {len(chosen_sections)} chosen passages — generating base-model rejected counterparts")

    # Initialize dpo_pairs if cells 1-4 were skipped
    if "dpo_pairs" not in vars():
        dpo_pairs = []

    for section in tqdm(chosen_sections, desc="Generating rejected"):
        user_prompt = random.choice(user_prompts)
        target_len = max(64, len(section["text"].split()))
        msgs = []
        if sys_prompt.strip():
            msgs.append({"role": "system", "content": sys_prompt})
        msgs.append({"role": "user", "content": user_prompt})
        response = openai_client.chat.completions.create(
            model=BASE_MODEL,
            messages=msgs,
            max_tokens=target_len + 50,
            temperature=0.85,
        )
        rejected_text = response.choices[0].message.content
        dpo_pairs.append({
            "system": sys_prompt,
            "user": user_prompt,
            "chosen": section["text"],
            "rejected": rejected_text,
        })

    print(f"Bootstrap added {len(chosen_sections)} pairs — total dpo_pairs: {len(dpo_pairs)}")

In [ ]:
# Format and write DPO training file
def format_dpo_sample(sys_prompt, user_prompt, chosen_text, rejected_text):
    messages = []
    if sys_prompt.strip():
        messages.append({"role": "system", "content": sys_prompt})
    messages.append({"role": "user", "content": user_prompt})
    return {
        "messages": messages,
        "chosen": [{"role": "assistant", "content": chosen_text}],
        "rejected": [{"role": "assistant", "content": rejected_text}],
    }

random.shuffle(dpo_pairs)
with open(TRAINING_FILENAME, "w") as f:
    for pair in dpo_pairs:
        sample = format_dpo_sample(pair["system"], pair["user"], pair["chosen"], pair["rejected"])
        f.write(json.dumps(sample) + "\n")

print(f"Wrote {len(dpo_pairs)} training examples to {TRAINING_FILENAME}")

# Sanity check: show one sample
with open(TRAINING_FILENAME) as f:
    sample = json.loads(f.readline())
print("\nSample chosen (first 200 chars):", sample["chosen"][0]["content"][:200])
print("Sample rejected (first 200 chars):", sample["rejected"][0]["content"][:200])

In [ ]:
import os
import tiktoken
from openai import OpenAI

PROJECT_ID = "proj_hUizl3mrZGSfmp4C6DI60dJo"
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def estimate_training_cost(training_file, model_name, n_epochs=3):
    with open(training_file) as f:
        lines = f.readlines()
    try:
        enc = tiktoken.encoding_for_model(model_name)
    except Exception:
        enc = tiktoken.get_encoding("cl100k_base")
    total_tokens = 0
    for line in lines:
        try:
            data = json.loads(line)
            for msg in data.get("messages", []) + data.get("chosen", []) + data.get("rejected", []):
                total_tokens += len(enc.encode(str(msg.get("content", ""))))
        except Exception:
            pass
    pricing = {"gpt-4o-mini": 3.00, "gpt-4o": 6.00, "gpt-3.5-turbo": 8.00}
    base = next((k for k in pricing if k in model_name), "gpt-4o-mini")
    price = pricing[base]
    total = total_tokens * n_epochs
    return {
        "examples": len(lines),
        "tokens_per_epoch": total_tokens,
        "n_epochs": n_epochs,
        "total_tokens": total,
        "price_per_1m": price,
        "estimated_cost_usd": round(price / 1_000_000 * total, 4),
    }

cost = estimate_training_cost(TRAINING_FILENAME, BASE_MODEL)
print(json.dumps(cost, indent=2))

In [ ]:
file_response = client.files.create(
    file=open(TRAINING_FILENAME, "rb"), purpose="fine-tune"
)
print("Uploaded:", file_response.id)

response = client.fine_tuning.jobs.create(
    model=BASE_MODEL,
    training_file=file_response.id,
    method={
        "type": "dpo",
        "dpo": {"hyperparameters": {"beta": DPO_BETA}},
    },
)
print(response)

In [ ]:
# Re-run to check progress
status_response = client.fine_tuning.jobs.retrieve(response.id)
print(f"STATUS: {status_response.status}")
print(f"MODEL ID: {status_response.fine_tuned_model}")
print(status_response)

In [ ]:
# Quick smoke test — edit the messages to match your project
from pprint import pprint

test_system = open(f"source_prompts/{PROJECT_NAME}/system_prompt.txt").read()
test_prompts = sorted([
    f for f in os.listdir(f"source_prompts/{PROJECT_NAME}/")
    if f.endswith(".txt") and f != "system_prompt.txt"
])
test_user = open(f"source_prompts/{PROJECT_NAME}/{test_prompts[0]}").read()

completion = client.chat.completions.create(
    model=status_response.fine_tuned_model,
    messages=[
        {"role": "system", "content": test_system},
        {"role": "user", "content": test_user},
    ],
    max_tokens=512,
    temperature=0.85,
)
pprint(completion.choices[0].message.content)

In [ ]:
# Save DPO model to the same generator db as SFT models
import datetime

experiments_str = ",".join(EXPERIMENT_NAMES) if EXPERIMENT_NAMES else "all"
with open(MODEL_RECORDS, "a") as f:
    f.write(
        f"{str(datetime.date.today())}\n"
        f"{PROJECT_NAME}-{experiments_str}-DPO\n"
        f"{status_response.fine_tuned_model}\n"
    )
print(f"Saved model {status_response.fine_tuned_model} to {MODEL_RECORDS}")